In [9]:
"""
NSW Job Training & Earnings — Causal Inference Analysis
========================================================
Based on: LaLonde (1986) and Dehejia & Wahba (1999)

WHAT THIS SCRIPT DOES (in plain English):
------------------------------------------
We want to know: did a 1970s job training program (NSW) actually raise
people's earnings?

We have two datasets:
  - exp : the real randomized experiment (RCT) — our gold standard
  - obs : treated people + a non-random CPS comparison group (harder case)

We run several methods on the harder "observational" data and check
whether each one can recover the true answer we know from the RCT.

HOW TO RUN:
-----------
  pip install causaldata econml
  python nsw_causal_analysis.py

OUTPUTS:
--------
  - Printed results for every method
  - plot_1_raw_distributions.png   : who are treated vs control?
  - plot_2_propensity_overlap.png  : do they have similar propensity scores?
  - plot_3_balance.png             : covariate balance before/after
  - plot_4_ate_summary.png         : all ATE estimates side by side
"""

# ── Standard library imports ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")   # keep output clean

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
matplotlib.use("Agg")               # saves plots to files (no pop-up window)
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegressionCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


# ══════════════════════════════════════════════════════════════════════════════
# STEP 0 — LOAD THE DATA
# ══════════════════════════════════════════════════════════════════════════════
# We load two things:
#   exp : the NSW randomized experiment (treated + randomly assigned controls)
#   obs : the "hard" observational dataset (treated + CPS survey controls)
#
# The CPS controls are regular people from a national survey — they were never
# in the program and are systematically different (older, more educated, etc.)
# This is what makes the observational problem hard.

def load_data():
    # Strategy 1: load the Dehejia-Wahba files directly from the original source.
    # This is the most reliable option and requires no special package.
    # Strategy 2: fall back to causaldata's mixtape versions if the download fails.

    try:
        print("  Trying to load Dehejia-Wahba data from NBER...")
        exp = pd.read_stata("https://users.nber.org/~rdehejia/data/nsw_dw.dta")
        cps = pd.read_stata("https://users.nber.org/~rdehejia/data/cps_controls.dta")
        print("  Loaded from NBER successfully.")

    except Exception:
        print("  NBER download failed. Falling back to causaldata package...")
        try:
            from causaldata import nsw_mixtape, cps_mixtape
            exp = nsw_mixtape.load_pandas().data
            cps = cps_mixtape.load_pandas().data
        except Exception:
            # Last resort: print what IS available so the user can fix it
            import causaldata
            available = [x for x in dir(causaldata) if not x.startswith("_")]
            raise ImportError(
                f"Could not load NSW data. Available datasets in causaldata: {available}\n"
                "Please check the dataset names above and update this function."
            )

    cps["treat"] = 0   # mark CPS respondents as "not treated"

    # Stata files load numeric columns as float64 — cast treat to int now
    # so every function downstream can safely use T.sum(), T==1, etc.
    exp["treat"] = exp["treat"].astype(int)
    cps["treat"] = cps["treat"].astype(int)

    # Build the observational dataset:
    #   treated people from the RCT  +  CPS controls
    # (we only keep columns that exist in both datasets)
    shared_cols = [c for c in exp.columns if c in cps.columns]
    obs = pd.concat([
        exp[exp["treat"] == 1][shared_cols],   # RCT treated
        cps[shared_cols]                        # CPS controls
    ], ignore_index=True)

    # Print a summary so we know what we loaded
    print("=" * 60)
    print("DATA LOADED")
    print("=" * 60)
    print(f"  Experimental sample : {len(exp)} people "
          f"({exp['treat'].sum()} treated, {(exp['treat']==0).sum()} control)")
    print(f"  Observational sample: {len(obs)} people "
          f"({obs['treat'].sum()} treated, {(obs['treat']==0).sum()} CPS controls)")

    return exp, obs


# The covariates (background characteristics) we use throughout
COVARIATES = ["age", "education", "black", "hispanic",
              "married", "nodegree", "re74", "re75"]
# re74 / re75 = real earnings in 1974 and 1975 (before the program)
# re78        = real earnings in 1978 (after the program) — our outcome


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — INSPECT THE DATA
# ══════════════════════════════════════════════════════════════════════════════
# Before any modelling, look at:
#   - Basic descriptive stats (means, min, max)
#   - Are there any missing values?
#   - How different are treated vs. control groups on each variable?
#     (This is the "selection bias" problem visualised)

def inspect_data(exp, obs):
    print("\n" + "=" * 60)
    print("STEP 1 — DATA INSPECTION")
    print("=" * 60)

    # 1a. Check for missing values in both datasets
    print("\n--- Missing values ---")
    print(f"  Experimental: {exp.isnull().sum().sum()} missing values")
    print(f"  Observational: {obs.isnull().sum().sum()} missing values")

    # 1b. Descriptive statistics for the experimental sample
    print("\n--- Experimental sample: means by treatment group ---")
    desc = exp.groupby("treat")[COVARIATES + ["re78"]].mean().T
    desc.columns = ["Control (RCT)", "Treated"]
    desc["Difference"] = desc["Treated"] - desc["Control (RCT)"]
    print(desc.round(2).to_string())

    # 1c. Descriptive statistics for the observational sample
    print("\n--- Observational sample: means by treatment group ---")
    desc2 = obs.groupby("treat")[COVARIATES + ["re78"]].mean().T
    desc2.columns = ["Control (CPS)", "Treated"]
    desc2["Difference"] = desc2["Treated"] - desc2["Control (CPS)"]
    print(desc2.round(2).to_string())
    print("\n  -> Notice how large the CPS differences are vs. the RCT differences.")
    print("     This is selection bias: CPS controls are not comparable people.")

    # 1d. Plot raw distributions: treated vs. CPS control for key variables
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    T = obs["treat"].values
    for i, col in enumerate(COVARIATES):
        axes[i].hist(obs.loc[T==0, col], bins=30, alpha=0.5,
                     color="#4878CF", label="CPS Control", density=True)
        axes[i].hist(obs.loc[T==1, col], bins=30, alpha=0.5,
                     color="#D65F5F", label="Treated", density=True)
        axes[i].set_title(col, fontsize=11, fontweight="bold")
        axes[i].set_ylabel("Density")
        axes[i].legend(fontsize=8)
    fig.suptitle("Step 1: Raw Distributions — Treated vs. CPS Controls\n"
                 "(Large differences = selection bias)", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig("plot_1_raw_distributions.png", dpi=150)
    plt.close(fig)
    print("\n  -> Saved: plot_1_raw_distributions.png")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — EXPERIMENTAL BENCHMARK (the true answer)
# ══════════════════════════════════════════════════════════════════════════════
# Since the NSW was a real randomized experiment, we can get the true causal
# effect just by comparing average earnings of treated vs. control.
# This is our TARGET — every method below is judged against this number.

def experimental_benchmark(exp):
    print("\n" + "=" * 60)
    print("STEP 2 — EXPERIMENTAL BENCHMARK (Gold Standard)")
    print("=" * 60)

    y1 = exp.loc[exp["treat"] == 1, "re78"]   # outcomes for treated
    y0 = exp.loc[exp["treat"] == 0, "re78"]   # outcomes for control

    # Simple difference in means
    ate = y1.mean() - y0.mean()

    # Standard error for the difference of two means
    se = np.sqrt(y1.var(ddof=1) / len(y1) + y0.var(ddof=1) / len(y0))

    # 95% confidence interval and p-value
    ci = (ate - 1.96 * se, ate + 1.96 * se)
    t_stat = ate / se
    pval = 2 * stats.t.sf(abs(t_stat), df=len(y1) + len(y0) - 2)

    print(f"  Treated avg. earnings  (re78): ${y1.mean():,.0f}  (n={len(y1)})")
    print(f"  Control avg. earnings  (re78): ${y0.mean():,.0f}  (n={len(y0)})")
    print(f"  ATE (true causal effect)     : ${ate:,.0f}")
    print(f"  Standard error               : ${se:,.0f}")
    print(f"  95% confidence interval      : (${ci[0]:,.0f}, ${ci[1]:,.0f})")
    print(f"  p-value                      : {pval:.4f}")
    print(f"\n  TARGET: all methods below should recover ~${ate:,.0f}")

    return {"label": "Experimental (RCT)", "ate": ate, "se": se, "ci": ci}


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — NAIVE OLS (shows the problem)
# ══════════════════════════════════════════════════════════════════════════════
# Ordinary Least Squares regression with covariates.
# On the RCT data it should match the benchmark (good sanity check).
# On the CPS observational data it will give a wildly wrong answer because
# the CPS controls are too different — this is selection bias in action.

def naive_ols(df, label):
    print(f"\n  Running OLS: {label}")

    X = df[COVARIATES].values
    T = df["treat"].values
    Y = df["re78"].values

    # Build the design matrix: intercept + treatment + covariates
    Xmat = np.column_stack([np.ones(len(T)), T, X])

    # Solve for OLS coefficients
    coef, _, _, _ = np.linalg.lstsq(Xmat, Y, rcond=None)

    # The treatment coefficient (index 1) is our ATE estimate
    ate = coef[1]

    # Heteroskedasticity-robust (HC0) standard error
    resid = Y - Xmat @ coef
    Xpinv = np.linalg.pinv(Xmat)
    cov   = Xpinv @ np.diag(resid ** 2) @ Xpinv.T
    se    = np.sqrt(cov[1, 1])
    ci    = (ate - 1.96 * se, ate + 1.96 * se)

    print(f"    ATE = ${ate:,.0f}  |  SE = ${se:,.0f}  |  "
          f"95% CI = (${ci[0]:,.0f}, ${ci[1]:,.0f})")

    return {"label": label, "ate": ate, "se": se, "ci": ci}


def run_ols_comparisons(exp, obs):
    print("\n" + "=" * 60)
    print("STEP 3 — NAIVE OLS (selection bias baseline)")
    print("=" * 60)
    print("  OLS on the RCT data should match the benchmark.")
    print("  OLS on the CPS data will be badly wrong (selection bias).")

    r_exp = naive_ols(exp, label="OLS on Experimental (RCT) data")
    r_obs = naive_ols(obs, label="OLS on Observational (CPS) data")

    print("\n  -> The CPS estimate is far from the benchmark.")
    print("     This shows why we need the methods below.")

    return r_exp, r_obs


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — PROPENSITY SCORE + OVERLAP
# ══════════════════════════════════════════════════════════════════════════════
# The propensity score is the probability of being treated given your
# background characteristics: P(treat=1 | age, education, re74, ...)
#
# We estimate it with logistic regression, then check "overlap":
# do treated and control units have similar propensity scores?
# Poor overlap means the groups are fundamentally incomparable.

def estimate_propensity_score(df):
    print("\n" + "=" * 60)
    print("STEP 4 — PROPENSITY SCORE + OVERLAP CHECK")
    print("=" * 60)

    X = df[COVARIATES].values
    T = df["treat"].values

    # Standardise covariates so logistic regression converges well
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Logistic regression with cross-validated regularisation
    model = LogisticRegressionCV(cv=5, max_iter=1000, random_state=42)
    model.fit(X_scaled, T)

    # Predicted probability of treatment for every person
    ps = model.predict_proba(X_scaled)[:, 1]

    # Print a quick summary of the propensity scores
    print(f"  Propensity scores for TREATED  — "
          f"mean={ps[T==1].mean():.3f}, "
          f"min={ps[T==1].min():.3f}, max={ps[T==1].max():.3f}")
    print(f"  Propensity scores for CONTROLS — "
          f"mean={ps[T==0].mean():.3f}, "
          f"min={ps[T==0].min():.3f}, max={ps[T==0].max():.3f}")
    print("  -> If these ranges barely overlap, matching/IPW will struggle.")

    # Plot: raw propensity score AND log-odds (logit) side by side
    logit_ps = np.log(ps / (1 - ps + 1e-9))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, vals, xlabel in zip(
        axes,
        [ps, logit_ps],
        ["Propensity score  P(treat=1|X)", "Log-odds  log[p/(1-p)]"]
    ):
        ax.hist(vals[T==0], bins=40, alpha=0.55,
                color="#4878CF", label="CPS Control", density=True)
        ax.hist(vals[T==1], bins=40, alpha=0.55,
                color="#D65F5F", label="Treated", density=True)
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel("Density")
        ax.set_title(f"Overlap: {xlabel}", fontsize=11, fontweight="bold")
        ax.legend()

    fig.suptitle("Step 4: Propensity Score Overlap\n"
                 "(Minimal overlap = hard to compare groups)",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig("plot_2_propensity_overlap.png", dpi=150)
    plt.close(fig)
    print("  -> Saved: plot_2_propensity_overlap.png")

    return ps


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — COVARIATE BALANCE (before AND after adjustment)
# ══════════════════════════════════════════════════════════════════════════════
# We measure balance using Standardised Mean Differences (SMD).
# SMD = (mean_treated - mean_control) / pooled_std
# Rule of thumb: |SMD| < 0.1 is considered "balanced".
#
# We check balance in three situations:
#   - Raw (unadjusted)
#   - After IPW weighting
#   - After propensity-score matching (1-NN with caliper 0.25 SD)

def compute_smd(X, T, weights_treated=None, weights_control=None):
    """
    Compute standardised mean difference for each covariate.
    If weights are given, compute weighted means.
    """
    smds = []
    for j in range(X.shape[1]):
        col = X[:, j]
        # Pooled standard deviation (always from raw data, not weighted)
        pooled_std = np.sqrt((col[T==1].var(ddof=1) + col[T==0].var(ddof=1)) / 2)
        if pooled_std == 0:
            smds.append(0.0)
            continue
        if weights_treated is not None:
            mean1 = np.average(col[T==1], weights=weights_treated)
            mean0 = np.average(col[T==0], weights=weights_control)
        else:
            mean1 = col[T==1].mean()
            mean0 = col[T==0].mean()
        smds.append((mean1 - mean0) / pooled_std)
    return np.array(smds)


def balance_analysis(obs, ps):
    print("\n" + "=" * 60)
    print("STEP 5 — COVARIATE BALANCE")
    print("=" * 60)

    X = obs[COVARIATES].values
    T = obs["treat"].values
    logit_ps = np.log(ps / (1 - ps + 1e-9))

    # ── Raw SMD (no adjustment) ───────────────────────────────────────────────
    smd_raw = compute_smd(X, T)

    # ── IPW-weighted SMD ──────────────────────────────────────────────────────
    # IPW gives each control unit a weight proportional to ps/(1-ps),
    # making the weighted control group look like the treated group.
    w_treated = np.ones(T.sum())
    w_control = ps[T==0] / (1 - ps[T==0])
    smd_ipw   = compute_smd(X, T, w_treated, w_control)

    # ── Post-matching SMD ─────────────────────────────────────────────────────
    # Match each treated unit to its nearest control (caliper 0.25 SD)
    cal_sd = 0.25 * logit_ps[T==0].std()
    nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
    nn.fit(logit_ps[T==0].reshape(-1, 1))
    dists, idx = nn.kneighbors(logit_ps[T==1].reshape(-1, 1))

    keep             = dists.flatten() <= cal_sd
    treated_idx      = np.where(T==1)[0][keep]
    matched_ctrl_idx = np.where(T==0)[0][idx.flatten()[keep]]

    X_matched = np.vstack([X[treated_idx], X[matched_ctrl_idx]])
    T_matched = np.array([1] * len(treated_idx) + [0] * len(matched_ctrl_idx))
    smd_matched = compute_smd(X_matched, T_matched)

    # ── Print the balance table ───────────────────────────────────────────────
    bal = pd.DataFrame({
        "Covariate"         : COVARIATES,
        "Raw SMD"           : smd_raw.round(3),
        "After IPW"         : smd_ipw.round(3),
        "After PSM (0.25s)" : smd_matched.round(3),
    })
    print("\n  Goal: |SMD| < 0.1 for each covariate after adjustment\n")
    print(bal.to_string(index=False))

    # ── Balance plot ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = np.arange(len(COVARIATES))

    ax.scatter(smd_raw,     y_pos - 0.2, color="#D65F5F", s=70,
               zorder=3, label="Raw (unadjusted)")
    ax.scatter(smd_ipw,     y_pos,       color="#4878CF", s=70,
               zorder=3, label="After IPW")
    ax.scatter(smd_matched, y_pos + 0.2, color="#2ecc71", s=70,
               zorder=3, label="After PSM (caliper 0.25s)")

    ax.axvline(0,    color="black", linewidth=0.8)
    ax.axvline( 0.1, color="grey",  linewidth=0.8, linestyle="--", alpha=0.7)
    ax.axvline(-0.1, color="grey",  linewidth=0.8, linestyle="--", alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(COVARIATES)
    ax.set_xlabel("Standardised Mean Difference (SMD)", fontsize=11)
    ax.set_title("Step 5: Covariate Balance Before and After Adjustment\n"
                 "(dashed lines = +/-0.1 threshold)", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig("plot_3_balance.png", dpi=150)
    plt.close(fig)
    print("\n  -> Saved: plot_3_balance.png")

    return bal


# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — PROPENSITY SCORE MATCHING (PSM)
# ══════════════════════════════════════════════════════════════════════════════
# For each treated person, find the most similar control person
# (the one with the closest propensity score).
#
# Caliper = a maximum distance threshold.
# Without it, some matches are very bad (a treated person matched to a very
# different control).  With caliper=0.25 SD, we drop those bad pairs.

def propensity_score_matching(obs, ps, caliper=None, label="PSM"):
    T = obs["treat"].values
    Y = obs["re78"].values
    logit_ps = np.log(ps / (1 - ps + 1e-9))

    treated_idx = np.where(T == 1)[0]
    control_idx = np.where(T == 0)[0]

    # Find 1 nearest neighbour in control group for each treated unit
    nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
    nn.fit(logit_ps[control_idx].reshape(-1, 1))
    dists, idx = nn.kneighbors(logit_ps[treated_idx].reshape(-1, 1))
    matched_ctrl_idx = control_idx[idx.flatten()]

    # Apply caliper: drop pairs where the match is too distant
    if caliper is not None:
        threshold        = caliper * logit_ps[control_idx].std()
        keep             = dists.flatten() <= threshold
        treated_idx      = treated_idx[keep]
        matched_ctrl_idx = matched_ctrl_idx[keep]
        n_dropped        = (~keep).sum()
    else:
        n_dropped = 0

    # ATT = average difference in outcomes across matched pairs
    diffs = Y[treated_idx] - Y[matched_ctrl_idx]
    att   = diffs.mean()
    se    = diffs.std(ddof=1) / np.sqrt(len(diffs))
    ci    = (att - 1.96 * se, att + 1.96 * se)

    print(f"  {label}: matched pairs={len(diffs)}, dropped={n_dropped}")
    print(f"    ATT = ${att:,.0f}  |  SE = ${se:,.0f}  |  "
          f"95% CI = (${ci[0]:,.0f}, ${ci[1]:,.0f})")

    return {"label": label, "ate": att, "se": se, "ci": ci}


def run_matching(obs, ps):
    print("\n" + "=" * 60)
    print("STEP 6 — PROPENSITY SCORE MATCHING")
    print("=" * 60)
    print("  Matching each treated person to their most similar control.")

    r1 = propensity_score_matching(obs, ps, caliper=None,
                                   label="PSM — no caliper")
    r2 = propensity_score_matching(obs, ps, caliper=0.25,
                                   label="PSM — caliper 0.25 SD")

    print("  -> Caliper version drops bad matches and usually gets closer "
          "to the benchmark.")
    return r1, r2


# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — INVERSE PROBABILITY WEIGHTING (IPW)
# ══════════════════════════════════════════════════════════════════════════════
# Instead of matching, we re-weight the sample.
# Treated units get weight 1/ps, controls get weight 1/(1-ps).
# This makes the weighted control group "look like" the treated group.
# We use the normalised (Hajek) version, which is more numerically stable.

def ipw(obs, ps, label="IPW (Normalized)"):
    print("\n" + "=" * 60)
    print("STEP 7 — INVERSE PROBABILITY WEIGHTING (IPW)")
    print("=" * 60)

    T = obs["treat"].values
    Y = obs["re78"].values

    # Normalised weights (Hajek estimator)
    w1  = T / ps
    w0  = (1 - T) / (1 - ps)
    ate = (Y * w1).sum() / w1.sum() - (Y * w0).sum() / w0.sum()

    # Warn about extreme weights (sign of poor overlap)
    max_w = max(w1[T==1].max(), w0[T==0].max())
    flag  = " <- WARNING: extreme weights suggest poor overlap" if max_w > 100 else " (OK)"
    print(f"  Largest IPW weight: {max_w:.1f}{flag}")

    # Bootstrap standard error (resample 500 times)
    rng   = np.random.default_rng(42)
    boots = []
    n     = len(T)
    for _ in range(500):
        idx  = rng.integers(0, n, n)
        _T, _Y, _ps = T[idx], Y[idx], ps[idx]
        _w1  = _T / _ps
        _w0  = (1 - _T) / (1 - _ps)
        boots.append(
            (_Y * _w1).sum() / _w1.sum() - (_Y * _w0).sum() / _w0.sum()
        )

    se = np.std(boots, ddof=1)
    ci = (ate - 1.96 * se, ate + 1.96 * se)

    print(f"  ATE = ${ate:,.0f}  |  Bootstrap SE = ${se:,.0f}  |  "
          f"95% CI = (${ci[0]:,.0f}, ${ci[1]:,.0f})")

    return {"label": label, "ate": ate, "se": se, "ci": ci}


# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — DOUBLY-ROBUST / AIPW
# ══════════════════════════════════════════════════════════════════════════════
# AIPW (Augmented IPW) combines:
#   1. An outcome model  — predicts earnings given background characteristics
#   2. A propensity model — predicts treatment probability
#
# It is "doubly robust": it gives the correct answer as long as AT LEAST ONE
# of the two models is correctly specified.  This makes it more reliable
# than IPW or matching alone.

def aipw(obs, ps, label="AIPW (Doubly-Robust)"):
    print("\n" + "=" * 60)
    print("STEP 8 — DOUBLY-ROBUST / AIPW")
    print("=" * 60)

    X = obs[COVARIATES].values
    T = obs["treat"].values
    Y = obs["re78"].values
    n = len(T)

    # Fit a linear outcome model separately for treated and controls
    mu1_model = LinearRegression().fit(X[T==1], Y[T==1])
    mu0_model = LinearRegression().fit(X[T==0], Y[T==0])

    # Predict potential outcomes for everyone
    mu1 = mu1_model.predict(X)   # predicted earnings if treated
    mu0 = mu0_model.predict(X)   # predicted earnings if not treated

    # AIPW score combines the outcome model prediction with an IPW correction
    psi1 = mu1 + T * (Y - mu1) / ps
    psi0 = mu0 + (1 - T) * (Y - mu0) / (1 - ps)

    ate = (psi1 - psi0).mean()
    se  = (psi1 - psi0).std(ddof=1) / np.sqrt(n)
    ci  = (ate - 1.96 * se, ate + 1.96 * se)

    print(f"  ATE = ${ate:,.0f}  |  SE = ${se:,.0f}  |  "
          f"95% CI = (${ci[0]:,.0f}, ${ci[1]:,.0f})")

    return {"label": label, "ate": ate, "se": se, "ci": ci}


# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — DOUBLE / DEBIASED ML (DML)
# ══════════════════════════════════════════════════════════════════════════════
# DML is the modern machine-learning version of AIPW.
# Instead of linear models for the nuisance functions, it uses flexible ML
# (gradient boosting here) and cross-fitting to avoid overfitting bias.
# Requires: pip install econml
# This step runs automatically if econml is installed; skips gracefully if not.

def dml(obs, label="Double/Debiased ML"):
    print("\n" + "=" * 60)
    print("STEP 9 — DOUBLE / DEBIASED ML (DML)")
    print("=" * 60)

    try:
        from econml.dml import LinearDML
        from sklearn.ensemble import (GradientBoostingRegressor,
                                      GradientBoostingClassifier)
    except ImportError:
        print("  econml not installed.  Run:  pip install econml")
        print("  Skipping DML — all other results are unaffected.")
        return None

    X = obs[COVARIATES].values
    T = obs["treat"].values
    Y = obs["re78"].values

    # LinearDML uses ML to partial out the effect of X on both Y and T,
    # then estimates the treatment effect on the residuals (cross-fitted).
    est = LinearDML(
        model_y=GradientBoostingRegressor(n_estimators=100, random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        random_state=42,
    )
    est.fit(Y, T, X=X)

    ate_inf = est.ate_inference(X=X)
    ate     = ate_inf.mean_point
    ci      = ate_inf.mean_ci(alpha=0.05)
    se      = (ci[1] - ci[0]) / (2 * 1.96)

    print(f"  ATE = ${ate:,.0f}  |  SE approx ${se:,.0f}  |  "
          f"95% CI = (${ci[0]:,.0f}, ${ci[1]:,.0f})")

    return {"label": label, "ate": ate, "se": se, "ci": ci}


# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — SENSITIVITY ANALYSIS (Rosenbaum Bounds)
# ══════════════════════════════════════════════════════════════════════════════
# All methods above assume "selection on observables" — that we have measured
# all the important confounders.  But what if we missed something?
#
# Rosenbaum bounds ask: how strong would unmeasured confounding need to be
# to make our result disappear?
#
# Gamma = 1  means no hidden confounding at all.
# Gamma = 2  means an unobserved factor could make one person twice as likely
#            to be treated as another identical person.
# We report the p-value upper bound at each Gamma.
# A result that stays significant up to Gamma = 3 or 4 is considered robust.

def rosenbaum_sensitivity(obs, ps, gammas=None):
    print("\n" + "=" * 60)
    print("STEP 10 — SENSITIVITY ANALYSIS (Rosenbaum Bounds)")
    print("=" * 60)
    print("  How much hidden confounding would it take to kill our result?")
    print("  Gamma=1: assumes no hidden confounding")
    print("  Gamma=2: an unobserved factor could double the odds of treatment")

    if gammas is None:
        gammas = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

    T = obs["treat"].values
    Y = obs["re78"].values
    logit_ps = np.log(ps / (1 - ps + 1e-9))

    # Re-do 1-NN matching (no caliper, for sensitivity purposes)
    nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
    nn.fit(logit_ps[T==0].reshape(-1, 1))
    _, idx       = nn.kneighbors(logit_ps[T==1].reshape(-1, 1))
    matched_ctrl = np.where(T==0)[0][idx.flatten()]

    diffs   = Y[np.where(T==1)[0]] - Y[matched_ctrl]
    n_pairs = len(diffs)
    pos     = (diffs > 0).sum()   # pairs where treated person earned more

    print(f"\n  Matched pairs: {n_pairs}  |  "
          f"Pairs where treated earned more: {pos}\n")
    print(f"  {'Gamma':>5}  {'p-value upper bound':>20}  Interpretation")
    print(f"  {'-'*58}")

    for g in gammas:
        p_plus  = g / (1 + g)
        mu_plus = n_pairs * p_plus
        sigma   = np.sqrt(n_pairs * p_plus * (1 - p_plus))
        z       = (pos - 0.5 - mu_plus) / sigma
        pval    = stats.norm.sf(z)
        note    = ("still significant (p<0.05)"
                   if pval < 0.05 else "no longer significant")
        print(f"  {g:>5.1f}  {pval:>20.4f}  {note}")

    print("\n  -> A large Gamma before losing significance = robust result.")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — SUMMARY PLOT (all estimates side by side)
# ══════════════════════════════════════════════════════════════════════════════

def summary_plot(results):
    print("\n" + "=" * 60)
    print("STEP 11 — SUMMARY PLOT")
    print("=" * 60)

    fig, ax = plt.subplots(figsize=(10, max(5, len(results) * 0.85)))

    for i, r in enumerate(results):
        # Colour-code by method type
        if "RCT" in r["label"]:
            color = "#2ecc71"    # green  = gold standard
        elif "OLS" in r["label"] and "CPS" in r["label"]:
            color = "#e74c3c"    # red    = known-biased naive estimate
        else:
            color = "#3498db"    # blue   = observational methods

        err_lo = r["ate"] - r["ci"][0]
        err_hi = r["ci"][1] - r["ate"]
        ax.errorbar(r["ate"], i,
                    xerr=[[err_lo], [err_hi]],
                    fmt="o", color=color, capsize=5,
                    markersize=9, linewidth=2, elinewidth=1.5)
        ax.text(r["ate"], i + 0.28, f"${r['ate']:,.0f}",
                ha="center", fontsize=8.5, color=color, fontweight="bold")

    # Vertical line showing the RCT benchmark
    rct_ate = next(r["ate"] for r in results if "RCT" in r["label"])
    ax.axvline(rct_ate, color="#2ecc71", linestyle="--", linewidth=1.2,
               alpha=0.8, label=f"RCT benchmark (${rct_ate:,.0f})")
    ax.axvline(0, color="grey", linestyle=":", linewidth=0.8)

    ax.set_yticks(range(len(results)))
    ax.set_yticklabels([r["label"] for r in results], fontsize=9.5)
    ax.set_xlabel("Estimated ATE / ATT  (1978 earnings, USD)", fontsize=11)
    ax.set_title(
        "NSW Job Training — All ATE Estimates with 95% CI\n"
        "(green = RCT target  |  red = biased naive OLS  |  blue = corrected methods)",
        fontsize=11, fontweight="bold"
    )
    ax.invert_yaxis()
    ax.legend(fontsize=9, loc="lower right")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig("plot_4_ate_summary.png", dpi=150)
    plt.close(fig)
    print("  -> Saved: plot_4_ate_summary.png")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN — runs every step in order
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("  NSW JOB TRAINING — CAUSAL INFERENCE ANALYSIS")
    print("  LaLonde (1986) / Dehejia & Wahba (1999)")
    print("=" * 60)

    # Step 0: Load data
    exp, obs = load_data()

    # Step 1: Inspect the data (descriptive stats + distribution plots)
    inspect_data(exp, obs)

    # Step 2: True experimental benchmark (our target number)
    result_rct = experimental_benchmark(exp)

    # Step 3: Naive OLS — shows how badly selection bias distorts things
    result_ols_exp, result_ols_obs = run_ols_comparisons(exp, obs)

    # Step 4: Estimate propensity scores + check overlap
    ps = estimate_propensity_score(obs)

    # Step 5: Covariate balance (raw / after IPW / after matching)
    balance_analysis(obs, ps)

    # Step 6: Propensity score matching (with and without caliper)
    result_psm_nocal, result_psm_cal = run_matching(obs, ps)

    # Step 7: Inverse Probability Weighting (IPW)
    result_ipw = ipw(obs, ps)

    # Step 8: Doubly-Robust / AIPW
    result_aipw = aipw(obs, ps)

    # Step 9: Double/Debiased ML (auto-skips if econml is not installed)
    result_dml = dml(obs)

    # Step 10: Rosenbaum sensitivity analysis
    rosenbaum_sensitivity(obs, ps)

    # Step 11: Summary plot of all ATE estimates
    all_results = [
        result_rct,
        result_ols_exp,
        result_ols_obs,
        result_psm_nocal,
        result_psm_cal,
        result_ipw,
        result_aipw,
    ]
    if result_dml is not None:
        all_results.append(result_dml)

    summary_plot(all_results)

    # Print a clean final table
    print("\n" + "=" * 60)
    print("FINAL SUMMARY — All ATE Estimates")
    print("=" * 60)
    print(f"  {'Method':<42} {'ATE':>8}   95% CI")
    print(f"  {'-'*68}")
    for r in all_results:
        print(f"  {r['label']:<42} ${r['ate']:>7,.0f}   "
              f"(${r['ci'][0]:,.0f}, ${r['ci'][1]:,.0f})")

    print("\nAll done! Check the four plot_*.png files saved in this folder.")


  NSW JOB TRAINING — CAUSAL INFERENCE ANALYSIS
  LaLonde (1986) / Dehejia & Wahba (1999)
  Trying to load Dehejia-Wahba data from NBER...
  Loaded from NBER successfully.
DATA LOADED
  Experimental sample : 445 people (185 treated, 260 control)
  Observational sample: 16177 people (185 treated, 15992 CPS controls)

STEP 1 — DATA INSPECTION

--- Missing values ---
  Experimental: 0 missing values
  Observational: 0 missing values

--- Experimental sample: means by treatment group ---
           Control (RCT)      Treated   Difference
age            25.049999    25.820000     0.760000
education      10.090000    10.350000     0.260000
black           0.830000     0.840000     0.020000
hispanic        0.110000     0.060000    -0.050000
married         0.150000     0.190000     0.040000
nodegree        0.830000     0.710000    -0.130000
re74         2107.030029  2095.570068   -11.450000
re75         1266.910034  1532.060059   265.149994
re78         4554.799805  6349.140137  1794.339966



In [10]:

    # ══════════════════════════════════════════════════════════════════════════
    # BONUS STEP — RESTRICTED CPS ANALYSIS
    # ══════════════════════════════════════════════════════════════════════════
    # Key question: what if we only kept CPS controls who look like the
    # treated group in terms of pre-program earnings?
    #
    # The treated group had mean re74 = $2,096 and mean re75 = $1,532.
    # Most CPS controls earned $14,000+ — completely incomparable.
    #
    # We try three increasingly tight restrictions on the CPS controls:
    #   - re74 < $10,000  (moderate restriction)
    #   - re74 < $5,000   (tight restriction)
    #   - re74 < $5,000 AND re75 < $5,000  (tightest — both years)
    #
    # For each restriction we rerun all methods and compare to the benchmark.
 
    print("\n" + "=" * 60)
    print("BONUS — RESTRICTED CPS ANALYSIS")
    print("=" * 60)
    print("  Restricting CPS controls to those with low pre-program earnings")
    print("  to test whether better overlap recovers the benchmark.")
 
    # The treated group for building the restricted obs dataset
    treated_rows = obs[obs["treat"] == 1]
    cps_rows     = obs[obs["treat"] == 0]
 
    restrictions = [
        ("re74 < $10,000",              cps_rows[cps_rows["re74"] < 10000]),
        ("re74 < $5,000",               cps_rows[cps_rows["re74"] < 5000]),
        ("re74 < $5,000 & re75 < $5,000", cps_rows[(cps_rows["re74"] < 5000) &
                                                     (cps_rows["re75"] < 5000)]),
    ]
 
    restricted_results = []
 
    for label, cps_subset in restrictions:
        obs_r = pd.concat([treated_rows, cps_subset], ignore_index=True)
        n_ctrl = len(cps_subset)
 
        print(f"\n  --- Restriction: {label} ---")
        print(f"  CPS controls remaining: {n_ctrl} (from 15,992)")
 
        if n_ctrl < 50:
            print("  Too few controls — skipping.")
            continue
 
        # Re-estimate propensity scores on restricted sample
        ps_r = estimate_propensity_score(obs_r)
 
        # Run all methods on restricted sample
        r_ols  = naive_ols(obs_r, label=f"OLS ({label})")
        r_psm  = propensity_score_matching(obs_r, ps_r, caliper=0.25,
                                           label=f"PSM 0.25σ ({label})")
        r_ipw  = ipw(obs_r, ps_r, label=f"IPW ({label})")
        r_aipw = aipw(obs_r, ps_r, label=f"AIPW ({label})")
 
        restricted_results.append({
            "restriction": label,
            "n_ctrl": n_ctrl,
            "ols":  r_ols["ate"],
            "psm":  r_psm["ate"],
            "ipw":  r_ipw["ate"],
            "aipw": r_aipw["ate"],
        })
 
    # ── Comparison plot: full CPS vs restricted ───────────────────────────────
    if restricted_results:
        print("\n  Building comparison plot...")
 
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        benchmark = result_rct["ate"]
 
        # Left plot: full CPS results
        ax = axes[0]
        full_labels  = ["OLS", "PSM (0.25σ)", "IPW", "AIPW"]
        full_values  = [result_ols_obs["ate"], result_psm_cal["ate"],
                        result_ipw["ate"],     result_aipw["ate"]]
        colors = ["#e74c3c" if abs(v - benchmark) > 500 else "#2ecc71"
                  for v in full_values]
        bars = ax.barh(full_labels, full_values, color=colors, alpha=0.8, edgecolor="white")
        ax.axvline(benchmark, color="#2ecc71", linestyle="--", linewidth=2,
                   label=f"RCT benchmark (${benchmark:,.0f})")
        ax.axvline(0, color="grey", linestyle=":", linewidth=0.8)
        ax.set_title("Full CPS (15,992 controls)\n← All methods fail", fontsize=12, fontweight="bold")
        ax.set_xlabel("ATE Estimate (1978 earnings, USD)")
        ax.legend(fontsize=9)
        ax.grid(axis="x", alpha=0.3)
        for bar, val in zip(bars, full_values):
            ax.text(val + (200 if val >= 0 else -200), bar.get_y() + bar.get_height()/2,
                    f"${val:,.0f}", va="center", ha="left" if val >= 0 else "right",
                    fontsize=10, fontweight="bold")
 
        # Right plot: best restricted result (tightest restriction)
        ax = axes[1]
        best = restricted_results[-1]   # tightest restriction
        rest_labels = ["OLS", "PSM (0.25σ)", "IPW", "AIPW"]
        rest_values = [best["ols"], best["psm"], best["ipw"], best["aipw"]]
        colors2 = ["#2ecc71" if abs(v - benchmark) < 800 else "#3498db"
                   for v in rest_values]
        bars2 = ax.barh(rest_labels, rest_values, color=colors2, alpha=0.8, edgecolor="white")
        ax.axvline(benchmark, color="#2ecc71", linestyle="--", linewidth=2,
                   label=f"RCT benchmark (${benchmark:,.0f})")
        ax.axvline(0, color="grey", linestyle=":", linewidth=0.8)
        ax.set_title(f"Restricted CPS ({best['n_ctrl']} controls)\n"
                     f"({best['restriction']})", fontsize=12, fontweight="bold")
        ax.set_xlabel("ATE Estimate (1978 earnings, USD)")
        ax.legend(fontsize=9)
        ax.grid(axis="x", alpha=0.3)
        for bar, val in zip(bars2, rest_values):
            ax.text(val + (200 if val >= 0 else -200), bar.get_y() + bar.get_height()/2,
                    f"${val:,.0f}", va="center", ha="left" if val >= 0 else "right",
                    fontsize=10, fontweight="bold")
 
        fig.suptitle(
            "Bonus: Does Restricting the CPS Comparison Group Help?\n"
            "Full CPS (left) vs. Low-Earnings CPS Subsample (right)",
            fontsize=13, fontweight="bold"
        )
        fig.tight_layout()
        fig.savefig("plot_5_restricted_cps.png", dpi=150)
        plt.close(fig)
        print("  -> Saved: plot_5_restricted_cps.png")
 
        # Print summary table
        print("\n" + "=" * 60)
        print("RESTRICTED CPS SUMMARY")
        print("=" * 60)
        print(f"  RCT benchmark: ${benchmark:,.0f}")
        print(f"\n  {'Restriction':<35} {'N ctrl':>7} {'OLS':>8} "
              f"{'PSM':>8} {'IPW':>8} {'AIPW':>8}")
        print(f"  {'-'*80}")
        print(f"  {'Full CPS (no restriction)':<35} {'15,992':>7} "
              f"${result_ols_obs['ate']:>7,.0f} "
              f"${result_psm_cal['ate']:>7,.0f} "
              f"${result_ipw['ate']:>7,.0f} "
              f"${result_aipw['ate']:>7,.0f}")
        for r in restricted_results:
            print(f"  {r['restriction']:<35} {r['n_ctrl']:>7,} "
                  f"${r['ols']:>7,.0f} ${r['psm']:>7,.0f} "
                  f"${r['ipw']:>7,.0f} ${r['aipw']:>7,.0f}")
 


BONUS — RESTRICTED CPS ANALYSIS
  Restricting CPS controls to those with low pre-program earnings
  to test whether better overlap recovers the benchmark.

  --- Restriction: re74 < $10,000 ---
  CPS controls remaining: 5895 (from 15,992)

STEP 4 — PROPENSITY SCORE + OVERLAP CHECK
  Propensity scores for TREATED  — mean=0.034, min=0.029, max=0.036
  Propensity scores for CONTROLS — mean=0.030, min=0.027, max=0.036
  -> If these ranges barely overlap, matching/IPW will struggle.
  -> Saved: plot_2_propensity_overlap.png

  Running OLS: OLS (re74 < $10,000)
    ATE = $1,243  |  SE = $630  |  95% CI = ($9, $2,476)
  PSM 0.25σ (re74 < $10,000): matched pairs=185, dropped=0
    ATT = $1,357  |  SE = $725  |  95% CI = ($-65, $2,779)

STEP 7 — INVERSE PROBABILITY WEIGHTING (IPW)
  Largest IPW weight: 34.2 (OK)
  ATE = $-1,450  |  Bootstrap SE = $596  |  95% CI = ($-2,619, $-281)

STEP 8 — DOUBLY-ROBUST / AIPW
  ATE = $1,138  |  SE = $512  |  95% CI = ($134, $2,142)

  --- Restriction: re74 <